# Build ordered speaker prompts

Run after transcript preparation. The model receives three annotated images plus groups in saved page/group order, including OCR text and boxes. It outputs one speaker label per line. Prepared targets and image caches remain unchanged.


In [6]:
from pathlib import Path
import hashlib
import json

# Find the project whether Jupyter starts in the workspace or notebooks folder.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'TaskData').is_dir() or (p / 'manga_notebook_codebase/TaskData').is_dir())
if not (ROOT / 'TaskData').is_dir():
    ROOT = ROOT / 'manga_notebook_codebase'
ROOT = ROOT.resolve()
PREP_ROOT = ROOT / 'outputs/transcript_prep_vl'
OUTPUT_ROOT = ROOT / 'outputs/speaker_prompts'
SPLITS = ('train', 'val')  # A one-item tuple needs the trailing comma.


## 1. One shared prompt

List groups in saved page/group order with IDs, OCR text, and boxes. The assistant returns one speaker label per line in that same order. Group IDs stay in the input only.


In [7]:
import sys
import importlib
sys.path.insert(0, str(ROOT / 'src'))
import vl_prompts
importlib.reload(vl_prompts)  # Pick up edits even in an existing kernel.
from vl_prompts import (OUTPUT_FORMAT, ordered_groups, build_speaker_prompt,
                        build_speaker_target, parse_speaker_output, permute_speaker_labels)


## 2. Load prepared records and keep answers separate

`prompts.jsonl` stores three image paths, the prompt, and the ordered `requested_groups`. `scoring_targets.jsonl` keeps accepted reference speakers separate. Training requests only accepted groups, with one speaker line per requested group. Character labels are shuffled during collation, not in these saved files.


In [3]:
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()
            if line.strip()]


def write_jsonl(path, records):
    path.write_text(''.join(json.dumps(record, ensure_ascii=False) + '\n'
                            for record in records), encoding='utf-8')


def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def build_split(split):
    if split not in ('train', 'val'):
        raise ValueError('split must be train or val')
    prepared = PREP_ROOT / split
    inputs_path, targets_path = prepared / 'inputs.jsonl', prepared / 'targets.jsonl'
    inputs, targets = read_jsonl(inputs_path), read_jsonl(targets_path)
    input_ids = [r['sequence_id'] for r in inputs]
    target_ids = [r['sequence_id'] for r in targets]
    if len(set(input_ids)) != len(input_ids) or len(set(target_ids)) != len(target_ids):
        raise ValueError('Duplicate sequence IDs')
    if set(input_ids) != set(target_ids):
        raise ValueError('Input and target sequence IDs differ')
    saved_split = json.loads((ROOT / 'outputs/split.json').read_text(encoding='utf-8'))
    if not set(input_ids).issubset(set(saved_split[split])):
        raise ValueError('Prepared sequence belongs to the wrong saved split')
    targets_by_id = {r['sequence_id']: r for r in targets}
    prompts, references, training, skipped = [], [], [], []
    image_hashes = {}
    for record in inputs:
        sid, pages = record['sequence_id'], record['pages']
        requested_groups = ordered_groups(pages)
        prompt = build_speaker_prompt(requested_groups)
        images = []
        for page in pages:
            image = (prepared / page['image']).resolve()
            if not image.is_relative_to(prepared.resolve()) or not image.is_file():
                raise ValueError(f'Missing or unsafe annotated image: {image}')
            images.append(str(image))
            image_hashes[str(image.relative_to(prepared.resolve()))] = file_hash(image)
        input_record = dict(sequence_id=sid, images=images, prompt=prompt,
                            requested_groups=requested_groups, output_format=OUTPUT_FORMAT)
        prompts.append(input_record)
        group_ids = {g['id'] for page in pages for g in page['groups']}
        target_pages = targets_by_id[sid]['pages']
        if len(target_pages) != 3:
            raise ValueError(f'{sid}: targets must have three page lists')
        answer = {}
        for index, page_targets in enumerate(target_pages):
            for target in page_targets:
                gid, speaker = target['group_id'], target['speaker']
                if (gid not in group_ids or not gid.startswith(f'p{index}_') or gid in answer):
                    raise ValueError(f'{sid}: invalid target group {gid}')
                if (speaker not in {'NARRATION', 'UNKNOWN'} and not (isinstance(speaker, str)
                        and speaker.startswith('char') and speaker[4:].isdigit()
                        and int(speaker[4:]) > 0)):
                    raise ValueError(f'{sid}: invalid accepted speaker {speaker}')
                answer[gid] = speaker
        references.append(dict(sequence_id=sid, speakers=answer,
                               scored_group_ids=list(answer)))
        if split == 'train':
            if answer:
                # Ask only for reliably labelled groups, keeping all three images.
                # This avoids teaching the model to omit requested groups.
                # Filter the ordered list, so target lines follow prompt order exactly.
                requested = [group for group in requested_groups if group['group_id'] in answer]
                labelled = [dict(group, speaker=answer[group['group_id']]) for group in requested]
                training.append(dict(
                    sequence_id=sid, images=images,
                    prompt=build_speaker_prompt(requested), requested_groups=requested,
                    assistant_response=build_speaker_target(labelled), output_format=OUTPUT_FORMAT))
            else:
                skipped.append(sid)
    output = OUTPUT_ROOT / split
    output.mkdir(parents=True, exist_ok=True)
    write_jsonl(output / 'prompts.jsonl', prompts)
    write_jsonl(output / 'scoring_targets.jsonl', references)
    if split == 'train':
        write_jsonl(output / 'training_examples.jsonl', training)
    # Save the shared template code and source/image hashes for reproducibility.
    notebook = ROOT / 'notebooks/build_speaker_prompt_Vl.ipynb'
    sources = [c['source'] for c in json.loads(notebook.read_text(encoding='utf-8'))['cells']
               if c['cell_type'] == 'code']
    files = ['prompts.jsonl', 'scoring_targets.jsonl']
    if split == 'train':
        files.append('training_examples.jsonl')
    summary = dict(output_format=OUTPUT_FORMAT, split=split, sequences=len(prompts), training_examples=len(training),
                   skipped_training_sequence_ids=skipped,
                   accepted_targets=sum(len(r['speakers']) for r in references),
                   source_sha256={'inputs': file_hash(inputs_path), 'targets': file_hash(targets_path),
                                  'split': file_hash(ROOT / 'outputs/split.json')},
                   notebook_code_sha256=hashlib.sha256(json.dumps(sources).encode()).hexdigest(),
                   shared_prompt_sha256=file_hash(ROOT / 'src/vl_prompts.py'),
                   annotated_image_sha256=image_hashes,
                   output_sha256={name: file_hash(output / name) for name in files})
    (output / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    return summary


## 3. Build available splits

This cell writes files when you run it. A missing split is skipped so validation preparation does not depend on preparing training data first. It does not run the preceding notebook automatically.


In [4]:
summaries = {}
for split in SPLITS:
    required = [PREP_ROOT / split / name for name in ('inputs.jsonl', 'targets.jsonl')]
    if not all(path.is_file() for path in required):
        print(f'Skipping {split}: run transcript preparation for this split first.')
        continue
    summaries[split] = build_split(split)
    print(json.dumps(summaries[split], indent=2))


{
  "output_format": "ordered_speaker_lines_v1",
  "split": "train",
  "sequences": 64,
  "training_examples": 63,
  "skipped_training_sequence_ids": [
    "seq_b3247b4b3a2f591b"
  ],
  "accepted_targets": 1277,
  "source_sha256": {
    "inputs": "da9cd4ef679896344b63b4c7a5e749ad586ddd847caf064c5f497672c2be7815",
    "targets": "6cfdd08475fab1d9b356c227f30c06c95a66aeaa0a53d8f1c9f8028adb189b9b",
    "split": "1f3a380537f410558ba5307bd4195531a9c0f20a354104a8861c137e3747422f"
  },
  "notebook_code_sha256": "316f866b6f19e2e230a224b16d7003848e033d85f2cb9649741140288ce6592d",
  "shared_prompt_sha256": "ff903453f7769ca3f02ce7966ebc829b48cbe31b9ec066b2339e94491944aa66",
  "annotated_image_sha256": {
    "annotated_images\\seq_71033fec4de494fc\\01.png": "3bbe74f85be44ec39e1000c1f5ecb9a0a1372924d0bffda9857eb2aea7387a2a",
    "annotated_images\\seq_71033fec4de494fc\\02.png": "ef2c82709728b1086d32f61dbea927c47a359edaa7b1d557aadc9dd5617f9974",
    "annotated_images\\seq_71033fec4de494fc\\03.png": "

## 4. Preview the exact model prompt

The preview contains OCR text and instructions only. Training and validation use the same function; only training examples contain an assistant response. Character labels are sequence-local identities, so scoring should account for a consistent renaming of character IDs rather than assuming the model chooses the reference numbering.


In [5]:
for split in summaries:
    records = read_jsonl(OUTPUT_ROOT / split / 'prompts.jsonl')
    if records:
        print(f"{split}: {records[0]['sequence_id']}")
        print(records[0]['prompt'])
        print('Image attachment order:', records[0]['images'])
        break


train: seq_71033fec4de494fc
You are given three consecutive manga pages with OCR groups marked on the images.
Images are in page order. p0, p1, and p2 refer to the first, second, and third page.
Boxes use original-image pixel coordinates [left, top, right, bottom].
Your task is only speaker attribution. Use the supplied OCR; do not transcribe it.
For every listed group, output exactly one speaker label.
Output exactly one label per line, in exactly the same order as the groups below.
Do not output group IDs, JSON, explanations, or extra text.
Use char1, char2, char3, ... for characters.
The same character must use the same label across all three pages.
Different characters must use different labels.
Character numbering is arbitrary.
char1 does NOT mean the first character, main character, or most common speaker.
Use NARRATION only for narrator text.
Use UNKNOWN only when the speaker cannot be determined.

Groups:
p0_g0: text="THIS I5 BAD! IT'S GOING TO BITE ME!" box=[299.0, 134.0, 388.